## D1 project: BANKING77 customer-support assistant


Build a **prototype that reads a customer’s message, identifies one of the 77 BANKING77 intents, and drafts a safe response**. For one relevant intent, it can demonstrate a **simulated transaction-status lookup**. It must not access a bank account, perform a transaction, or claim that a real banking action has occurred.


**Example**


**Input:** “My card payment is still pending. What happened?”

**Output:** A validated JSON object containing the predicted BANKING77 intent, a helpful response that does not invent an account status, and—only if appropriate—a request to use the *mock* status-lookup tool.

This choice covers the assignment’s classification, generation, structured-output, tool-use, evaluation, and responsible-use requirements in one coherent application.

In [1]:
!pip -q install datasets pandas

In [2]:
from datasets import load_dataset

base = "https://huggingface.co/datasets/PolyAI/banking77/resolve/main/data"
banking = load_dataset(
    "parquet",
    data_files={
        "train":    f"{base}/train-00000-of-00001.parquet",
        "test":     f"{base}/test-00000-of-00001.parquet",
        },
)


print(banking)
print(banking['train'][0])
print(banking['test'][0])

labels_names = banking['train'].features['label'].names
print("no_of_intents:", len(labels_names))
print("first_examples_intent:", labels_names[banking["train"][0]["label"]])

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}
{'text': 'How do I locate my card?', 'label': 11}
no_of_intents: 77
first_examples_intent: card_arrival


In [3]:
from collections import Counter

label_names = banking["train"].features["label"].names

for split in ["train", "test"]:
    counts = Counter(banking[split]["label"])
    empty_messages = sum(
        not text.strip() for text in banking[split]["text"]
    )

    print(f"\n{split.upper()}")
    print("Messages:", len(banking[split]))
    print("Intents represented:", len(counts))
    print("Fewest examples in an intent:", min(counts.values()))
    print("Most examples in an intent:", max(counts.values()))
    print("Empty messages:", empty_messages)

print("\nFive example messages:")
for index in [0, 100, 500, 1000, 2000]:
    row = banking["train"][index]
    print(f"{index}: [{label_names[row['label']]}] {row['text']}")


TRAIN
Messages: 10003
Intents represented: 77
Fewest examples in an intent: 35
Most examples in an intent: 187
Empty messages: 0

TEST
Messages: 3080
Intents represented: 77
Fewest examples in an intent: 40
Most examples in an intent: 40
Empty messages: 0

Five example messages:
0: [card_arrival] I am still waiting on my card?
100: [card_arrival] can you share card tracking number?
500: [card_payment_wrong_exchange_rate] Why is the exchange rate different from what I expected?
1000: [fiat_currency_support] What are the currency types that I can keep money in.
2000: [pending_top_up] My top is still pending.


In [4]:
from collections import defaultdict
import random

SEED = 42

def balanced_indices(split, examples_per_intent, seed):
    by_intent = defaultdict(list)

    for index, label in enumerate(split["label"]):
        by_intent[label].append(index)

    rng = random.Random(seed)
    selected = []
    for label in range(len(by_intent)):
        selected.extend(
        rng.sample(by_intent[label], examples_per_intent)
        )
    return sorted(selected)

dev_indices = balanced_indices(banking["train"], 1, SEED)
final_indices = balanced_indices(banking["train"], 2, SEED)


dev_set = banking["train"].select(dev_indices)
final_set = banking["train"].select(final_indices)

assert len(dev_set) == 77
assert len(final_set) == 154
assert len(dev_set["label"]) == 77
assert len(final_set["label"]) == 154

print("seed:", SEED)
print("dev set length:", len(dev_set))
print("final set length:", len(final_set))
print(" first 5 dev indices:", dev_indices[:5])
print(" first 5 final indices:", final_indices[:5])


seed: 42
dev set length: 77
final set length: 154
 first 5 dev indices: [108, 160, 381, 533, 658]
 first 5 final indices: [107, 139, 154, 224, 321]


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [6]:
!pip -q install -U transformers accelerate

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.float16,
    device_map = "auto",
)

print("loaded:", model_name)
print("model_device:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded: Qwen/Qwen2.5-1.5B-Instruct
model_device: cuda:0


 the tokenizer converts text into token IDs; the pretrained causal model generates the next tokens; float16 reduces GPU memory use; and device_map="auto" places the model on the available device. The first run downloads model files, so it may take a few minutes.

In [8]:
def baseline_reply(message):
    prompt = f"Identify the banking intent and respond to the customer:\n{message}"

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,  # Greedy decoding: reproducible baseline
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


for index in [0, 25, 50]:
    row = dev_set[index]
    print(f"\n--- Development example {index} ---")
    print("Customer:", row["text"])
    print("Expected intent:", label_names[row["label"]])
    print("Baseline output:", baseline_reply(row["text"]))



--- Development example 0 ---
Customer: Is it possible for me to track the card is was mailed to me?
Expected intent: card_arrival
Baseline output: Yes, you can usually track the mailing of your bank card through your account or contact information provided when you received the card. If you have access to this information, you should be able to check its delivery status online. Alternatively, if you need further assistance with tracking, please provide more details about how you received the card (e.g., physical mail, email) so I can guide you accordingly.

--- Development example 25 ---
Customer: What is the status of my top up?
Expected intent: top_up_reverted
Baseline output: I'm sorry, but I cannot provide an accurate response without more information about your specific situation. Could you please provide me with some additional details such as the name of your bank or the type of account you have? This will help me better understand your request and provide you with a more appr

These outputs give us three useful baseline findings:

**Example 0**: Helpful general guidance, but no exact card_arrival label. It also suggests account-based tracking without knowing whether that feature exists.

**Example 25**: The model correctly avoids claiming it knows the top-up status. However, the message alone does not clearly establish top_up_reverted; that label may be impossible to infer reliably from this wording. Keep this as an ambiguity case, not simply a model mistake.

**Example 50**: It describes an intent instead of naming verify_source_of_funds, then starts listing unsupported possibilities. The output also appears cut off at the 120-token limit.

Next, test one zero-shot prompt. **“Zero-shot”** means giving instructions and allowed labels but no worked examples. Keep the same three development messages and greedy decoding:

In [9]:
def zero_shot_reply(message):
    labels = ", ".join(label_names)
    prompt = (
        "Choose the single best BANKING77 intent for the customer message "
        "from the allowed labels below. Then write a brief, general reply. "
        "Do not claim to know the customer's account status.\n\n"
        f"Allowed labels: {labels}\n\n"
        f"Customer message: {message}\n\n"
        "Use this format:\nIntent: <exact allowed label>\nReply: <brief reply>"
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


for index in [0, 25, 50]:
    row = dev_set[index]
    print(f"\n--- Development example {index} ---")
    print("Customer:", row["text"])
    print("Expected intent:", label_names[row["label"]])
    print("Zero-shot output:", zero_shot_reply(row["text"]))


--- Development example 0 ---
Customer: Is it possible for me to track the card is was mailed to me?
Expected intent: card_arrival
Zero-shot output: Intent: get_physical_card
Reply: We can help you track your physical card that has been mailed to you. Please provide us with your tracking number or any other relevant information so we can assist further.

--- Development example 25 ---
Customer: What is the status of my top up?
Expected intent: top_up_reverted
Zero-shot output: Intent: top_up_status
Reply: We can help you check your top-up status. Please provide us with your account number or any other relevant information so we can assist you further.

--- Development example 50 ---
Customer: Where do my funds originate from?
Expected intent: verify_source_of_funds
Zero-shot output: Intent: verify_source_of_funds
Reply: To help you understand where your funds come from, we need to verify the source of your funds. Please provide us with any additional information or documents that may 

The zero-shot prompt improved the format, but exposed important problems:

Example	What happened
0	| get_physical_card is a valid label, but card_arrival fits a mailed card more closely. The reply promises tracking help we have not implemented.
25	top_up_status is not a BANKING77 label. The reply also asks for an account number in chat—unsafe for this prototype. The underlying message remains ambiguous.
50	The label is correct, but the reply asks for verification documents without a secure, authorized process.

Record these in your experiment log. A model printing “Intent:” does not mean the intent is valid; we will validate it in Python later.

Now test the constraint-based prompt. This changes the instructions while keeping the model, three messages, and greedy decoding the same:

In [10]:
def constrained_reply(message):
    labels = ", ".join(label_names)
    prompt = (
        "You are a prototype banking-support assistant. "
        "Select ONE intent whose spelling exactly matches an allowed label. "
        "Do not invent a new label.\n"
        "Write a brief reply following these rules:\n"
        "- Never claim to see an account, transaction, card delivery, or top-up status.\n"
        "- Never promise that a banking action or lookup has been performed.\n"
        "- Do not ask for account numbers, passwords, PINs, tracking numbers, "
        "or identity documents in chat.\n"
        "- If the message lacks details, acknowledge what cannot be determined "
        "and direct the customer to official secure support.\n"
        "- Refer sensitive or consequential actions to a human agent.\n\n"
        f"Allowed labels: {labels}\n"
        f"Customer message: {message}\n\n"
        "Output exactly two lines:\n"
        "Intent: <one exact allowed label>\n"
        "Reply: <brief safe reply>"
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


for index in [0, 25, 50]:
    row = dev_set[index]
    print(f"\n--- Development example {index} ---")
    print("Expected intent:", label_names[row["label"]])
    print("Constrained output:", constrained_reply(row["text"]))


--- Development example 0 ---
Expected intent: card_arrival
Constrained output: Intent: card_delivery_estimate
Reply: We can provide you with an estimated delivery date once your card arrives. Please keep an eye on your mailbox or check our website for updates.

--- Development example 25 ---
Expected intent: top_up_reverted
Constrained output: Intent: top_up_status
Reply: Your top-up request is currently being processed. We will update you once it's complete.

--- Development example 50 ---
Expected intent: verify_source_of_funds
Constrained output: Intent: verify_source_of_funds
Reply: We need to verify your source of funds to ensure it's legitimate. Please provide additional information if possible.


Before trying another prompt, add a small **Python validation gate**. Prompt rules are requests to the model; validation is code that can reject an output.

In [11]:
def validate_two_line_output(raw_output, allowed_labels):
    lines = raw_output.strip().splitlines()

    if len(lines) != 2:
        raise ValueError(f"Expected exactly two lines; got {len(lines)}")
    if not lines[0].startswith("Intent: "):
        raise ValueError("Missing 'Intent: ' line")
    if not lines[1].startswith("Reply: "):
        raise ValueError("Missing 'Reply: ' line")

    intent = lines[0][len("Intent: "):].strip()
    reply = lines[1][len("Reply: "):].strip()

    if intent not in allowed_labels:
        raise ValueError(f"Unknown BANKING77 intent: {intent!r}")
    if not reply:
        raise ValueError("Reply is empty")

    return {"intent": intent, "reply": reply}


allowed_labels = set(label_names)

for index in [0, 25, 50]:
    row = dev_set[index]
    raw = constrained_reply(row["text"])

    print(f"\nExample {index}; expected: {label_names[row['label']]}")
    try:
        result = validate_two_line_output(raw, allowed_labels)
        print("Valid format and label:", result["intent"])
        print("Correct intent:", result["intent"] == label_names[row["label"]])
        print("Reply for manual safety review:", result["reply"])
    except ValueError as error:
        print("REJECTED:", error)


Example 0; expected: card_arrival
Valid format and label: card_delivery_estimate
Correct intent: False
Reply for manual safety review: We can provide you with an estimated delivery date once your card arrives. Please keep an eye on your mailbox or check our website for updates.

Example 25; expected: top_up_reverted
REJECTED: Unknown BANKING77 intent: 'top_up_status'

Example 50; expected: verify_source_of_funds
Valid format and label: verify_source_of_funds
Correct intent: True
Reply for manual safety review: We need to verify your source of funds to ensure it's legitimate. Please provide additional information if possible.


**What this checks:** exact two-line format, a real dataset label, and a nonempty reply.

**What it cannot check:** whether the label is correct or the reply makes a false banking claim. That is why the cell prints those separately for review. We’ll later replace this temporary text format with validated JSON and add a safety/handoff step.

Run it and paste the three results. In particular, check whether example 25 is rejected rather than quietly accepted.

In [13]:
def role_context_reply(message):

    system_message = (
        "You are a prototype first-line banking support assistant, not a bank employee "
        "with account access. Your task is to identify the closest BANKING77 intent "
        "and give general guidance. You cannot see account, payment, or delivery status; "
        "do not state or imply that you can. Do not invent bank policies, ask for "
        "sensitive information in chat, or claim an action was completed. "
        "For account-specific or consequential requests, direct the customer to "
        "official secure support or a human agent.\n\n"
        f"Allowed intent labels: {', '.join(label_names)}\n\n"
        "Respond in exactly two lines:\n"
        "Intent: <one exact allowed label>\n"
        "Reply: <brief general guidance>"
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_message},
         {"role": "user", "content": message}
         ],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )


for index in [0, 25, 50]:
    row = dev_set[index]
    raw = role_context_reply(row["text"])
    print(f"\nExample {index}; expected: {label_names[row['label']]}")
    print("Raw Output:", raw)

try:
  result = validate_two_line_output(raw, allowed_labels)
  print("Valid format and label:", result["intent"])
  print("Correct intent:", result["intent"] == label_names[row["label"]])
except ValueError as error:
  print("REJECTED:", error)


Example 0; expected: card_arrival
Raw Output: Intent: get_physical_card
Reply: We recommend checking your mailbox or any other method of delivery confirmation provided by the shipping company to locate your physical card. If you're still unable to find it, please contact our customer service for further assistance.

Example 25; expected: top_up_reverted
Raw Output: Intent: top_up_status
Reply: I will check the status of your top-up and provide you with the latest update shortly. Please stay tuned.

Example 50; expected: verify_source_of_funds
Raw Output: Intent: verify_source_of_funds
Reply: To help us understand where your funds come from, we need to verify the source of your funds. Please provide any additional details or documents required by our verification process.
Valid format and label: verify_source_of_funds
Correct intent: True


The role prompt did not solve the core problems. Example 0 still confuses a mailed card with obtaining a physical card. Example 25 again invents top_up_status and falsely promises a lookup. Example 50 gets the label right but refers to an unverified “verification process.” This is strong evidence for your report that a system-role instruction alone cannot guarantee valid labels or safe replies.

Next we’ll build the few-shot prompt, but first choose its examples carefully. Few-shot means showing the model a few labeled training messages before asking it to classify a new message. Run this cell to inspect candidate examples from training data only, excluding your development examples:

In [14]:
candidate_labels = [
    "card_arrival",
    "card_delivery_estimate",
    "get_physical_card",
    "top_up_reverted",
    "pending_top_up",
    "verify_source_of_funds",
]

for name in candidate_labels:
    label_id = label_names.index(name)
    matches = [
        i for i, label in enumerate(banking["train"]["label"])
        if label == label_id and i not in dev_indices
    ]

    print(f"\n{name}:")
    for i in matches[:2]:
        print(f"  train index {i}: {banking['train'][i]['text']}")


card_arrival:
  train index 0: I am still waiting on my card?
  train index 1: What can I do if my card still hasn't arrived after 2 weeks?

card_delivery_estimate:
  train index 1006: Can it specifically be delivered on a certain date?
  train index 1007: I need to get my card quickly

get_physical_card:
  train index 3994: I'm not sure what to do about the PIN, because I don't have one yet.
  train index 3995: Is my PIN sent separably?

top_up_reverted:
  train index 3155: My top up did not show up as shown and my money has disappeared, can you help me fix this glitch problem?
  train index 3156: Has my top-up been cancelled?

pending_top_up:
  train index 1928: How long does a top-up take to go through?
  train index 1929: I am under the impression that my top up is still pending

verify_source_of_funds:
  train index 6481: where exactly does money come from
  train index 6482: Where did my money come from?


These examples show why we inspected the data first: the two get_physical_card candidates discuss a PIN, so they would not help distinguish “getting a card” from “tracking a mailed card.” We’ll leave them out.

For the few-shot experiment, use four training examples that contrast card arrival vs delivery date and reverted vs pending top-up. The assistant replies below are our safe prototype wording, not answers supplied by BANKING77.

In [15]:
few_shot_examples = [
    (0, "I cannot see your card's delivery status. Please use official secure support if it has not arrived."),
    (1006, "I cannot confirm a delivery date. Please check the official card-delivery information."),
    (3156, "I cannot see whether a top-up was cancelled. A human agent can review it through official secure support."),
    (1929, "I cannot see your top-up status. Please check it through official secure support."),
]

assert all(i not in dev_indices for i, _ in few_shot_examples)

few_shot_system = (
    "You are a prototype first-line banking support assistant, not a bank employee "
    "with account access. Identify the closest BANKING77 intent and give brief, "
    "general guidance. Do not invent account facts, promise a lookup or completed "
    "action, or request sensitive information in chat. Refer account-specific "
    "questions to official secure support.\n\n"
    f"Allowed intent labels: {', '.join(label_names)}\n\n"
    "Respond in exactly two lines:\n"
    "Intent: <one exact allowed label>\n"
    "Reply: <brief general guidance>"
)

def few_shot_reply(message):
    messages = [{"role": "system", "content": few_shot_system}]

    for train_index, safe_reply in few_shot_examples:
        row = banking["train"][train_index]
        intent = label_names[row["label"]]
        messages.append({"role": "user", "content": row["text"]})
        messages.append({
            "role": "assistant",
            "content": f"Intent: {intent}\nReply: {safe_reply}"
        })

    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

allowed_labels = set(label_names)

for index in [0, 25, 50]:
    row = dev_set[index]
    raw = few_shot_reply(row["text"])
    print(f"\nExample {index}; expected: {label_names[row['label']]}")
    print("Raw Output:", raw)

    try:
        result = validate_two_line_output(raw, allowed_labels)
        print("Correct intent:", result["intent"] == label_names[row["label"]])
    except ValueError as error:
        print("REJECTED:", error)



Example 0; expected: card_arrival
Raw Output: Intent: card_arrival
Reply: I cannot provide tracking details for physical cards. You should check the official mail delivery service for updates.
Correct intent: True

Example 25; expected: top_up_reverted
Raw Output: Intent: top_up_status
Reply: I don't have real-time updates for individual transactions. You should check your account details via official secure support.
REJECTED: Unknown BANKING77 intent: 'top_up_status'

Example 50; expected: verify_source_of_funds
Raw Output: Intent: verify_source_of_funds
Reply: To verify the source of funds, please contact official secure support for assistance.
Correct intent: True


What changed: the model now sees four demonstrations before the customer message. The model, development inputs, and greedy decoding stay the same. This completes the fourth E2 prompt variant; we can compare all four after seeing its results.

Paste the three outputs. In particular, we should not expect few-shot prompting to reveal an unknown top-up status from example 25’s ambiguous wording.

Few-shot prompting improved example 0 (card_arrival) and kept example 50 correct. It also avoided the false promise to check a transaction. But example 25 still produced the invented label top_up_status, which your validator correctly rejected. That is a useful conclusion: few-shot examples help, but they do not enforce the label set.

Three examples are enough to illustrate behavior, not enough to choose a winning prompt. Next, compare all four prompts on the same 15 development messages: the three we studied plus 12 selected reproducibly from the remaining development set. This will run 60 generations, so allow some time.

In [19]:
import random
import pandas as pd

rng = random.Random(42)
comparison_indices = sorted(
    [0, 25, 50] +
    rng.sample([i for i in range(len(dev_set)) if i not in {0, 25, 50}], 12)
)

prompt_variants = {
    "zero_shot": zero_shot_reply,
    "constrained": constrained_reply,
    "role_context": role_context_reply,
    "few_shot": few_shot_reply,

}

records = []

for index in comparison_indices:
  row = dev_set[index]
  expected = label_names[row["label"]]

  for variant, generate_reply in prompt_variants.items():
    raw = generate_reply(row["text"])

    try:
      result = validate_two_line_output(raw, allowed_labels)
      predicted = result["intent"]
      validation_error = None
    except ValueError as error:
      predicted = None
      validation_error = str(error)

    records.append({
            "dev_index": index,
            "customer_message": row["text"],
            "expected_intent": expected,
            "prompt_variant": variant,
            "raw_output": raw,
            "predicted_intent": predicted,
            "valid_output": validation_error is None,
            "correct_intent": predicted == expected,
            "validation_error": validation_error,
    })

results = pd.DataFrame(records)

summary = results.groupby("prompt_variant").agg(
    messages=("dev_index", "count"),
    valid_outputs=("valid_output", "sum"),
    correct_intents=("correct_intent", "sum"),
)
summary["valid_rate"] = summary["valid_outputs"] / summary["messages"]
summary["intent_accuracy"] = summary["correct_intents"] / summary["messages"]

print("Development indices:", comparison_indices)
print(summary)


Development indices: [0, 4, 5, 6, 12, 14, 15, 18, 25, 30, 33, 37, 50, 57, 75]
                messages  valid_outputs  correct_intents  valid_rate  \
prompt_variant                                                         
constrained           15             12                2    0.800000   
few_shot              15             11                6    0.733333   
role_context          15             11                2    0.733333   
zero_shot             15             10                3    0.666667   

                intent_accuracy  
prompt_variant                   
constrained            0.133333  
few_shot               0.400000  
role_context           0.133333  
zero_shot              0.200000  


On this 15-message development sample, few-shot has the highest intent accuracy: 6/15 (40%). The constrained prompt has the highest format/label validity: 12/15 (80%), but only 2/15 correct intents. These are different measures, and the sample is too small to claim that either prompt is generally best.

Before decoding experiments, inspect why the few-shot prompt missed nine messages. Save the results first—Colab runtimes can expire—then print the mistakes:

In [20]:
results.to_csv("banking77_prompt_comparison_dev.csv", index=False)

few_shot_results = results[results["prompt_variant"] == "few_shot"]

print("Few-shot mistakes:")
print(
    few_shot_results.loc[
        ~few_shot_results["correct_intent"],
        ["dev_index", "customer_message", "expected_intent",
         "predicted_intent", "validation_error", "raw_output"],
    ].to_string(index=False, max_colwidth=100)
)

Few-shot mistakes:
 dev_index                                                                                     customer_message                     expected_intent                   predicted_intent                                                validation_error                                                                                           raw_output
         4 A pending charge of 1L appears on my statement.  Please explain as I have not made a purchase for...           extra_charge_on_statement                  declined_transfer                                                            None Intent: declined_transfer\nReply: It seems there might be an issue with a pending transfer. You s...
         6                                                How many different currencies can I have my money in?               fiat_currency_support     supported_cards_and_currencies                                                            None Intent: supported_cards_and_currencies

These mistakes reveal **three different problems**, so changing decoding alone will not solve them:

**Wrong but valid labels**: examples 4, 6, 33, and 57 confuse related banking topics.

**Insufficient evidence in the message:** examples 15 (“Do you take SWIFT transfers?”), 25 (“status of my top up?”), and 37 (“what is the word?”) do not clearly support their assigned labels. Record these as dataset/ambiguity limitations; don’t force the model to guess hidden information.

**Reliability and safety:** example 75 breaks the two-line format and may invent identity-verification steps. In example 33, “I cannot see any issues” could misleadingly imply that a check occurred.

For E3 (decoding), use few-shot as the current development candidate because it had the most correct labels, not because it is production-safe. We’ll compare its greedy output with sampling while holding its prompt fixed.

First, make a reusable version of that same few-shot prompt:

In [22]:
def generate_few_shot(message, sample=False, seed=None):
    messages = [{"role": "system", "content": few_shot_system}]

    for train_index, safe_reply in few_shot_examples:
        row = banking["train"][train_index]
        messages.append({"role": "user", "content": row["text"]})
        messages.append({
            "role": "assistant",
            "content": (
                f"Intent: {label_names[row['label']]}\n"
                f"Reply: {safe_reply}"
            ),
        })

    messages.append({"role": "user", "content": message})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    settings = {
        "max_new_tokens": 160,
        "do_sample": sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if sample:
        torch.manual_seed(seed)
        settings.update(temperature=0.7, top_p=0.9)

    with torch.inference_mode():
        output = model.generate(**inputs, **settings)

    return tokenizer.decode(
        output[0, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

In [23]:
decoding_records = []

for index in [0, 25, 37, 75]:
    row = dev_set[index]

    for setting, sample, seed in [
        ("greedy", False, None),
        ("temperature_0.7_top_p_0.9_seed_42", True, 42),
        ("temperature_0.7_top_p_0.9_seed_43", True, 43),
    ]:
        raw = generate_few_shot(row["text"], sample=sample, seed=seed)

        try:
            parsed = validate_two_line_output(raw, allowed_labels)
            predicted = parsed["intent"]
            error = None
        except ValueError as exc:
            predicted = None
            error = str(exc)

        decoding_records.append({
            "dev_index": index,
            "expected_intent": label_names[row["label"]],
            "setting": setting,
            "predicted_intent": predicted,
            "validation_error": error,
            "raw_output": raw,
        })

decoding_results = pd.DataFrame(decoding_records)
decoding_results.to_csv("banking77_decoding_dev.csv", index=False)

print(decoding_results[
    ["dev_index", "expected_intent", "setting",
     "predicted_intent", "validation_error", "raw_output"]
].to_string(index=False, max_colwidth=100))

 dev_index                     expected_intent                           setting       predicted_intent                          validation_error                                                                                           raw_output
         0                        card_arrival                            greedy           card_arrival                                      None Intent: card_arrival\nReply: I cannot provide tracking details for physical cards. You should che...
         0                        card_arrival temperature_0.7_top_p_0.9_seed_42           card_arrival                                      None Intent: card_arrival\nReply: Unfortunately, tracking of mail deliveries is not available directly...
         0                        card_arrival temperature_0.7_top_p_0.9_seed_43 card_delivery_estimate                                      None Intent: card_delivery_estimate\nReply: Unfortunately, tracking details may not be available as we...
        25  

**Greedy** chooses the most likely next token each time. **Temperature** changes how sharply probabilities favor likely tokens; ** top-p** restricts sampling to a high-probability token pool. Two seeds let us see whether sampling changes the answer. Save/download the CSV, then paste the table; we’ll judge both variability and whether any output became less reliable.